In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [2]:
import os

print(os.listdir("/kaggle/input"))

['datasets']


In [3]:
print(os.listdir("/kaggle/input/datasets/sudhirkh"))

['deepfashion2', 'deepfashion-preprocessed-val', 'deepfashion-preprocessed']


In [4]:
print(os.listdir("/kaggle/working"))

['.virtual_documents']


In [5]:
DATA_ROOT = "/kaggle/input/datasets/sudhirkh/deepfashion2/DeepFashion2"

train_image_folder = DATA_ROOT + "/train/image"
train_json_folder  = DATA_ROOT + "/train/annos"

val_image_folder = DATA_ROOT + "/validation/image"
val_json_folder  = DATA_ROOT + "/validation/annos"

test_image_folder = DATA_ROOT + "/test/image"

In [6]:
import os

print(len(os.listdir(train_image_folder)))
print(len(os.listdir(train_json_folder)))

print(len(os.listdir(val_image_folder)))
print(len(os.listdir(val_json_folder)))

191961
191961
32153
32153


In [ ]:
import torch

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import json

sample = os.listdir(train_json_folder)[0]

with open(os.path.join(train_json_folder, sample)) as f:
    data = json.load(f)

print(data.keys())

In [ ]:
import os
import json
from collections import Counter
from tqdm import tqdm

category_count = Counter()

for file in tqdm(os.listdir(train_json_folder)):
    
    with open(os.path.join(train_json_folder, file)) as f:
        data = json.load(f)

    for key in data:
        if key.startswith("item"):
            cid = data[key]["category_id"]
            category_count[cid] += 1

print(category_count.most_common(10))

In [7]:
top5_ids = [1,8,7,2,9]

category_to_index = {
    cat:i for i,cat in enumerate(top5_ids)
}

In [ ]:
import os
import json
from collections import Counter
from tqdm import tqdm
image_paths = []
labels = []

for file in tqdm(os.listdir(train_json_folder)):

    with open(os.path.join(train_json_folder,file)) as f:
        data = json.load(f)

    multi_hot = [0]*5
    valid = False

    for key in data:
        if key.startswith("item"):

            cid = data[key]["category_id"]

            if cid in category_to_index:
                idx = category_to_index[cid]
                multi_hot[idx] = 1
                valid = True

    if valid:
        img = file.replace(".json",".jpg")
        img_path = os.path.join(train_image_folder,img)

        image_paths.append(img_path)
        labels.append(multi_hot)

print("Usable training images:",len(image_paths))

In [ ]:
import pickle

save_path = "/kaggle/working/train_data.pkl"

with open(save_path, "wb") as f:
    pickle.dump((image_paths, labels), f)

print("Saved successfully!")

In [3]:
import pickle

with open("/kaggle/input/datasets/sudhirkh/deepfashion-preprocessed/train_data.pkl", "rb") as f:
    image_paths, labels = pickle.load(f)

print("Loaded:", len(image_paths))

Loaded: 144174


In [ ]:
val_image_paths = []
val_labels = []

for file in tqdm(os.listdir(val_json_folder)):

    with open(os.path.join(val_json_folder, file)) as f:
        data = json.load(f)

    multi_hot = [0] * 5
    valid = False

    for key in data:
        if key.startswith("item"):

            cid = data[key]["category_id"]

            if cid in category_to_index:
                idx = category_to_index[cid]
                multi_hot[idx] = 1
                valid = True

    if valid:
        img = file.replace(".json", ".jpg")
        img_path = os.path.join(val_image_folder, img)

        val_image_paths.append(img_path)
        val_labels.append(multi_hot)

print("Validation images:", len(val_image_paths))

In [ ]:
save_path = "/kaggle/working/validation_data.pkl"

with open(save_path, "wb") as f:
    pickle.dump((val_image_paths, val_labels), f)

print("Saved successfully!")

In [4]:
import pickle

with open("/kaggle/input/datasets/sudhirkh/deepfashion-preprocessed-val/validation_data.pkl", "rb") as f:
    val_image_paths, val_labels = pickle.load(f)

print("Loaded:", len(val_image_paths))

Loaded: 23741


In [5]:
from sklearn.model_selection import train_test_split

train_paths, _, train_labels, _ = train_test_split(
    image_paths,
    labels,
    test_size = 0.5,
    random_state=42
)

print("Train size:",len(train_paths))

Train size: 72087


In [ ]:
train_paths = image_paths
train_labels = labels

print("Train size:", len(train_paths))

In [6]:
from torchvision import transforms
train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor()
])

val_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

In [7]:
import torch
from torch.utils.data import Dataset
from PIL import Image

class FashionDataset(Dataset):

    def __init__(self, image_paths, labels, transform=None):

        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):

        image = Image.open(self.image_paths[idx]).convert("RGB")

        label = torch.tensor(self.labels[idx], dtype=torch.float32)

        if self.transform:
            image = self.transform(image)

        return image, label

In [8]:
train_dataset = FashionDataset(
    train_paths,
    train_labels,
    transform=train_transform
)

In [9]:
val_dataset = FashionDataset(
    val_image_paths,
    val_labels,
    transform=val_transform
)

In [10]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)


In [11]:
val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

In [12]:
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights
weights = MobileNet_V2_Weights.DEFAULT
model = mobilenet_v2(weights=weights)

model

MobileNetV2(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU6(inplace=True)
    )
    (1): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU6(inplace=True)
        )
        (1): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (2): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (1): BatchNorm2d(96, eps=

In [17]:
import torch
import torch.nn as nn
from torchvision import models


model = models.efficientnet_b0(pretrained=True)

model

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B0_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B0_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 161MB/s]


EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivat

In [13]:
import torch.nn as nn

in_features = model.classifier[1].in_features
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.classifier = nn.Sequential(
    nn.Linear(in_features, 512),
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(512, 5)
)

model = model.to(device)

model

MobileNetV2(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU6(inplace=True)
    )
    (1): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU6(inplace=True)
        )
        (1): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (2): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (1): BatchNorm2d(96, eps=

In [14]:
criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4
)

In [15]:
print(len(train_loader))
print(len(val_loader))

2253
742


In [16]:
import torch
import numpy as np
from tqdm import tqdm
from sklearn.metrics import f1_score

epochs = 7
best_val_f1 = 0

SAVE_PATH = "/kaggle/working/best_model_mobilenet_full_dataset.pth"

for epoch in range(epochs):

    # ======================
    # TRAIN
    # ======================
    model.train()
    train_loss = 0
    train_preds = []
    train_labels_all = []

    for images, labels in tqdm(train_loader):

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

        # convert logits → probabilities
        probs = torch.sigmoid(outputs)

        preds = (probs > 0.5).int()

        train_preds.append(preds.cpu().numpy())
        train_labels_all.append(labels.cpu().numpy())

    train_loss /= len(train_loader)

    train_preds = np.vstack(train_preds)
    train_labels_all = np.vstack(train_labels_all)

    train_f1 = f1_score(train_labels_all, train_preds, average="macro")


    # ======================
    # VALIDATION
    # ======================
    model.eval()
    val_loss = 0
    val_preds = []
    val_labels_all = []

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()

            probs = torch.sigmoid(outputs)

            preds = (probs > 0.5).int()

            val_preds.append(preds.cpu().numpy())
            val_labels_all.append(labels.cpu().numpy())

    val_loss /= len(val_loader)

    val_preds = np.vstack(val_preds)
    val_labels_all = np.vstack(val_labels_all)

    val_f1 = f1_score(val_labels_all, val_preds, average="macro")


    # ======================
    # PRINT RESULTS
    # ======================
    print("\n---------------------------")
    print(f"Epoch {epoch+1}/{epochs}")
    print(f"Train Loss : {train_loss:.4f}")
    print(f"Train F1   : {train_f1:.4f}")
    print(f"Val Loss   : {val_loss:.4f}")
    print(f"Val F1     : {val_f1:.4f}")


    # ======================
    # SAVE BEST MODEL
    # ======================
    if val_f1 > best_val_f1:

        best_val_f1 = val_f1

        torch.save(model.state_dict(), SAVE_PATH)

        print("✅ Best model saved!")

100%|██████████| 2253/2253 [11:59<00:00,  3.13it/s]



---------------------------
Epoch 1/7
Train Loss : 0.3419
Train F1   : 0.7086
Val Loss   : 0.2774
Val F1     : 0.7858
✅ Best model saved!


100%|██████████| 2253/2253 [06:34<00:00,  5.71it/s]



---------------------------
Epoch 2/7
Train Loss : 0.2450
Train F1   : 0.8183
Val Loss   : 0.2511
Val F1     : 0.8129
✅ Best model saved!


100%|██████████| 2253/2253 [06:21<00:00,  5.91it/s]


...


---------------------------
Epoch 5/7
Train Loss : 0.1461
Train F1   : 0.9023
Val Loss   : 0.2625
Val F1     : 0.8231
✅ Best model saved!


100%|██████████| 2253/2253 [06:09<00:00,  6.10it/s]



---------------------------
Epoch 6/7
Train Loss : 0.1257
Train F1   : 0.9169
Val Loss   : 0.2857
Val F1     : 0.8166


100%|██████████| 2253/2253 [06:13<00:00,  6.03it/s]



---------------------------
Epoch 7/7
Train Loss : 0.1097
Train F1   : 0.9284
Val Loss   : 0.2921
Val F1     : 0.8233
✅ Best model saved!


In [ ]:
model.load_state_dict(torch.load("/kaggle/working/best_model.pth"))
model = model.to(device)
model.eval()

In [17]:
val_probs = []
val_preds = []
val_labels_all = []

with torch.no_grad():

    for images, labels in val_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        probs = torch.sigmoid(outputs)
        preds = (probs > 0.5).int()

        val_probs.append(probs.cpu().numpy())
        val_preds.append(preds.cpu().numpy())
        val_labels_all.append(labels.cpu().numpy())

val_probs = np.vstack(val_probs)
val_preds = np.vstack(val_preds)
val_labels_all = np.vstack(val_labels_all)

In [18]:
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

precision = precision_score(val_labels_all, val_preds, average="macro")
recall = recall_score(val_labels_all, val_preds, average="macro")
f1 = f1_score(val_labels_all, val_preds, average="macro")
roc_auc = roc_auc_score(val_labels_all, val_probs, average="macro")

print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)
print("ROC-AUC:", roc_auc)

Precision: 0.840345895545495
Recall: 0.8072270420125991
F1: 0.8232525508716305
ROC-AUC: 0.9501714828332892


In [19]:
for t in [0.3, 0.4, 0.5, 0.6]:

    preds = (val_probs > t).astype(int)

    f1 = f1_score(val_labels_all, preds, average="macro")

    print("Threshold", t, "→ Macro F1:", f1)

Threshold 0.3 → Macro F1: 0.8227391451512466
Threshold 0.4 → Macro F1: 0.8246659714910944
Threshold 0.5 → Macro F1: 0.8232525508716305
Threshold 0.6 → Macro F1: 0.8192006397266933


In [20]:
model = mobilenet_v2(weights=None)

In [21]:
model

MobileNetV2(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU6(inplace=True)
    )
    (1): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU6(inplace=True)
        )
        (1): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (2): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (1): BatchNorm2d(96, eps=

In [25]:
import torch
import torch.nn as nn
from torchvision import models


model = models.efficientnet_b0(pretrained=False)

model

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivat

**Training From Scratch**

In [26]:
model

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivat

In [22]:
import torch.nn as nn

in_features = model.classifier[1].in_features
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.classifier = nn.Sequential(
    nn.Linear(in_features, 512),
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(512, 5)
)

model = model.to(device)

model

MobileNetV2(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU6(inplace=True)
    )
    (1): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU6(inplace=True)
        )
        (1): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (2): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (1): BatchNorm2d(96, eps=

In [23]:
criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4
)

In [29]:
import torch
import numpy as np
from tqdm import tqdm
from sklearn.metrics import f1_score

epochs = 15
best_val_f1 = 0

SAVE_PATH = "/kaggle/working/best_model_efficient_net_from_scratch.pth"

for epoch in range(epochs):

    # ======================
    # TRAIN
    # ======================
    model.train()
    train_loss = 0
    train_preds = []
    train_labels_all = []

    for images, labels in tqdm(train_loader):

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

        # convert logits → probabilities
        probs = torch.sigmoid(outputs)

        preds = (probs > 0.5).int()

        train_preds.append(preds.cpu().numpy())
        train_labels_all.append(labels.cpu().numpy())

    train_loss /= len(train_loader)

    train_preds = np.vstack(train_preds)
    train_labels_all = np.vstack(train_labels_all)

    train_f1 = f1_score(train_labels_all, train_preds, average="macro")


    # ======================
    # VALIDATION
    # ======================
    model.eval()
    val_loss = 0
    val_preds = []
    val_labels_all = []

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()

            probs = torch.sigmoid(outputs)

            preds = (probs > 0.5).int()

            val_preds.append(preds.cpu().numpy())
            val_labels_all.append(labels.cpu().numpy())

    val_loss /= len(val_loader)

    val_preds = np.vstack(val_preds)
    val_labels_all = np.vstack(val_labels_all)

    val_f1 = f1_score(val_labels_all, val_preds, average="macro")


    # ======================
    # PRINT RESULTS
    # ======================
    print("\n---------------------------")
    print(f"Epoch {epoch+1}/{epochs}")
    print(f"Train Loss : {train_loss:.4f}")
    print(f"Train F1   : {train_f1:.4f}")
    print(f"Val Loss   : {val_loss:.4f}")
    print(f"Val F1     : {val_f1:.4f}")


    # ======================
    # SAVE BEST MODEL
    # ======================
    if val_f1 > best_val_f1:

        best_val_f1 = val_f1

        torch.save(model.state_dict(), SAVE_PATH)

        print("✅ Best model saved!")

100%|██████████| 2253/2253 [06:28<00:00,  5.80it/s]



---------------------------
Epoch 1/15
Train Loss : 0.5733
Train F1   : 0.2240
Val Loss   : 0.5435
Val F1     : 0.3194
✅ Best model saved!


100%|██████████| 2253/2253 [06:34<00:00,  5.72it/s]



---------------------------
Epoch 2/15
Train Loss : 0.5113
Train F1   : 0.3795
Val Loss   : 0.4972
Val F1     : 0.3947
✅ Best model saved!


100%|██████████| 2253/2253 [06:31<00:00,  5.75it/s]


...


---------------------------
Epoch 13/15
Train Loss : 0.2964
Train F1   : 0.7606
Val Loss   : 0.3741
Val F1     : 0.7014
✅ Best model saved!


100%|██████████| 2253/2253 [06:12<00:00,  6.05it/s]



---------------------------
Epoch 14/15
Train Loss : 0.2862
Train F1   : 0.7724
Val Loss   : 0.3904
Val F1     : 0.6958


100%|██████████| 2253/2253 [06:11<00:00,  6.07it/s]



---------------------------
Epoch 15/15
Train Loss : 0.2754
Train F1   : 0.7837
Val Loss   : 0.3706
Val F1     : 0.7141
✅ Best model saved!


In [24]:
import torch
import numpy as np
from tqdm import tqdm
from sklearn.metrics import f1_score

epochs = 20
best_val_f1 = 0

SAVE_PATH = "/kaggle/working/best_model_scratch_mobile_net.pth"

for epoch in range(epochs):

    # ======================
    # TRAIN
    # ======================
    model.train()
    train_loss = 0
    train_preds = []
    train_labels_all = []

    for images, labels in tqdm(train_loader):

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

        # convert logits → probabilities
        probs = torch.sigmoid(outputs)

        preds = (probs > 0.5).int()

        train_preds.append(preds.cpu().numpy())
        train_labels_all.append(labels.cpu().numpy())

    train_loss /= len(train_loader)

    train_preds = np.vstack(train_preds)
    train_labels_all = np.vstack(train_labels_all)

    train_f1 = f1_score(train_labels_all, train_preds, average="macro")


    # ======================
    # VALIDATION
    # ======================
    model.eval()
    val_loss = 0
    val_preds = []
    val_labels_all = []

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()

            probs = torch.sigmoid(outputs)

            preds = (probs > 0.5).int()

            val_preds.append(preds.cpu().numpy())
            val_labels_all.append(labels.cpu().numpy())

    val_loss /= len(val_loader)

    val_preds = np.vstack(val_preds)
    val_labels_all = np.vstack(val_labels_all)

    val_f1 = f1_score(val_labels_all, val_preds, average="macro")


    # ======================
    # PRINT RESULTS
    # ======================
    print("\n---------------------------")
    print(f"Epoch {epoch+1}/{epochs}")
    print(f"Train Loss : {train_loss:.4f}")
    print(f"Train F1   : {train_f1:.4f}")
    print(f"Val Loss   : {val_loss:.4f}")
    print(f"Val F1     : {val_f1:.4f}")


    # ======================
    # SAVE BEST MODEL
    # ======================
    if val_f1 > best_val_f1:

        best_val_f1 = val_f1

        torch.save(model.state_dict(), SAVE_PATH)

        print("✅ Best model saved!")

100%|██████████| 2253/2253 [06:12<00:00,  6.04it/s]



---------------------------
Epoch 1/20
Train Loss : 0.5845
Train F1   : 0.1880
Val Loss   : 0.5655
Val F1     : 0.2316
✅ Best model saved!


100%|██████████| 2253/2253 [06:05<00:00,  6.16it/s]



---------------------------
Epoch 2/20
Train Loss : 0.5382
Train F1   : 0.3242
Val Loss   : 0.5262
Val F1     : 0.3304
✅ Best model saved!


100%|██████████| 2253/2253 [06:10<00:00,  6.09it/s]


...


---------------------------
Epoch 18/20
Train Loss : 0.2882
Train F1   : 0.7733
Val Loss   : 0.4157
Val F1     : 0.6675
✅ Best model saved!


100%|██████████| 2253/2253 [06:24<00:00,  5.86it/s]



---------------------------
Epoch 19/20
Train Loss : 0.2762
Train F1   : 0.7853
Val Loss   : 0.4215
Val F1     : 0.6685
✅ Best model saved!


100%|██████████| 2253/2253 [06:35<00:00,  5.70it/s]



---------------------------
Epoch 20/20
Train Loss : 0.2676
Train F1   : 0.7966
Val Loss   : 0.4183
Val F1     : 0.6668
